In [3]:
from collections import Counter
import json, os, itertools

data_folder = "data/art_or_artifice/"

with open(f'{data_folder}humanannotationfinal.json') as f:
	data = json.load(f)


storymap = {'The Kingdom': 'The Kingdom That Failed','Returns':'Returns','MaintainenceH': 'Maintenance, Hvidovre','The Last Dance': 'The Last Dance with my Dad','Trash':'Trash','ListeningFC':'Listening For the Click','A Triangle': 'A Triangle', 'Keys': 'Keys','Barbara': 'Barbara,Detroit,1996','CertainEM': 'Certain European Movies','Beyond-Nat':'Beyond Nature','The Facade': "The Facade Renovation That_s Going Well" }

overall_ranks = {story_id: [] for story_id in storymap.values()}

for story in data:
    story_id = story['What is the story identifier (in the Google Doc)']
    story_id = story_id.replace("That's", "That_s").replace("Façade", "Facade")
    if "Facade" in story_id:
        story_id = "The Facade Renovation That_s Going Well"
    if 'Beyond-Nat' in story_id:
        story_id = 'Beyond Nature'
    elif '—' in story_id:
        story_id = storymap[story_id.split('—')[0]]
    elif '-' in story_id:
        story_id = storymap[story_id.split('-')[0]]

    with open(f'{data_folder}/aspectstories/{story_id}/sequencemap.json') as mp:
        order = {int(k): v for k,v in json.load(mp).items()}

    rank_texts = ["", "Most Preferred", "Second Most Preferred", "Third Most Preferred", "Least Favorite"]

    cleaned_ranks = {}
    for story_idx in range(1,5):
        model = order[story_idx]
        rank = rank_texts.index(story[f'Rank each of the four stories based on your preference [Story {story_idx}]'])
        cleaned_ranks[model] = rank

    overall_ranks[story_id].append(cleaned_ranks)

models = ["Claude.txt", "GPT3.5.txt", "NewYorker.txt", "GPT4.txt"]
AA_pairs = []
for story_id in overall_ranks:
    full_texts = {}
    for model in models:
        with open(f"{data_folder}/aspectstories/{story_id}/{model}") as f:
            full_texts[model] = f.read()

    for m1, m2 in itertools.combinations(models, 2):
        wins = []
        for clean_ranks in overall_ranks[story_id]:
            if clean_ranks[m1] < clean_ranks[m2]:
                wins.append(m1)
            else:
                wins.append(m2)
        win_counts = Counter(wins)
        winner, win_counts = win_counts.most_common(1)[0]
        win_counts /= len(wins)
        # print(f"{story_id.ljust(40)} {m1.ljust(15)} vs {m2.ljust(15)} {winner} {win_counts}")

        preference = "1" if winner == m1 else "2"
        sample = {"original_id": f"{story_id}_{m1}_{m2}", "model1": m1, "model2": m2, "story1": full_texts[m1], "story2": full_texts[m2], "winner": winner, "win_counts": win_counts, "preference": preference}
        AA_pairs.append(sample)

print(len(AA_pairs))
with open("data/art_or_artifice/AA_pairs_data.json", "w") as f:
    json.dump(AA_pairs, f, indent=4)


72


# Let's make the final version of the dataset with AB/BA

In [8]:
with open("data/art_or_artifice/AA_pairs_data.json", "r") as f:
    AA_pairs = json.load(f)
AA_pairs[0].keys()

dict_keys(['original_id', 'model1', 'model2', 'story1', 'story2', 'winner', 'win_counts', 'preference'])

In [9]:
import json

with open("data/art_or_artifice/AA_pairs_data.json", "r") as f:
    AA_pairs = json.load(f)

with open("prompts/pairwise_pref.txt", "r") as f:
    pairwise_prompt = f.read()

# Make AB/BA pairs
all_pairs_data = []
for sample in AA_pairs:
    sample1 = {"id": f"aspect_artartifice_{len(all_pairs_data)}", "original_id": f"{sample['original_id']}", "sample_type": "pairwise-art", "model1": sample["model1"], "model2": sample["model2"], "paragraph1": sample["story1"], "paragraph2": sample["story2"], "reference_preference": sample["preference"], "reference_win_counts": sample["win_counts"]}
    sample1["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", sample1["paragraph1"]).replace("[[PARAGRAPH2]]", sample1["paragraph2"])
    all_pairs_data.append(sample1)

    reference_preference2 = "2" if sample["preference"] == "1" else "1"
    sample2 = {"id": f"aspect_artartifice_{len(all_pairs_data)}", "original_id": f"{sample['original_id']}", "sample_type": "pairwise-art", "model1": sample["model2"], "model2": sample["model1"], "paragraph1": sample["story2"], "paragraph2": sample["story1"], "reference_preference": reference_preference2, "reference_win_counts": sample["win_counts"]}
    sample2["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", sample2["paragraph1"]).replace("[[PARAGRAPH2]]", sample2["paragraph2"])
    all_pairs_data.append(sample2)

with open("data/art_or_artifice/AA_pairs_data.json", "w") as f:
    json.dump(all_pairs_data, f, indent=4)


In [40]:
with open("data/lamp_PRGSH_aspect.json", "r") as f:
    lamp_PRGSH_aspect = json.load(f)

lamp_PRGSH_test += all_pairs_data

with open("data/lamp_PRGSH_test.json", "w") as f:
    json.dump(lamp_PRGSH_test, f, indent=4)

print(Counter(d['sample_type'] for d in lamp_PRGSH_test))

Counter({'pairwise-gold': 1206, 'pairwise-silver': 1120, 'pairwise-lmarena': 576, 'reward': 430, 'pairwise': 404, 'pairwise-h': 300, 'pairwise-P1': 215, 'pairwise-P2': 215, 'pairwise-P3': 209, 'pairwise-P4': 199, 'pairwise-P5': 183, 'pairwise-P6': 159, 'pairwise-art': 144, 'pairwise-P7': 138})


# TTCW descriptions

In [8]:
import os, json

ttcw_aspects = [
    {"aspect_id": 1, "category": "fluency", "aspect_name": "Narrative Pacing", "context": "Compression/stretching of time in fiction writing, also known as pacing, refers to the manipulation of time in storytelling for dramatic effect, pacing, or other narrative purposes. Essentially, it's about controlling the perceived speed and rhythm at which a story unfolds.", "question": "Does the manipulation of time in terms of compression or stretching feel appropriate and balanced?"},
    {"aspect_id": 2, "category": "fluency", "aspect_name": "Summary and Exposition", "context": "'Scene' and 'summary/exposition' are two crucial elements of narrative storytelling, and balancing them appropriately is an important skill in fiction writing.", "question": "Does the story display an awareness and insight into the balance between scene and summary/exposition?"},
    {"aspect_id": 3, "category": "fluency", "aspect_name": "Language Proficiency and Literary Devices", "context": "When a writer uses 'Idioms', 'Metaphors', and 'Literary Allusions', they are using literary devices to add depth, interest, and nuanced meaning to their work.", "question": "Does the story make sophisticated use of idiom, metaphor, and literary allusion?"},
    {"aspect_id": 4, "category": "fluency", "aspect_name": "Narrative Ending", "context": "If the writer ends the piece simply because they are 'tired of writing', the conclusion might feel abrupt, disjointed, or unfulfilling to the reader. It suggests a rushed ending, where plot threads might be left unresolved and character arcs incomplete.", "question": "Does the end of the story feel natural and earned, as opposed to arbitrary or abrupt?"},
    {"aspect_id": 5, "category": "fluency", "aspect_name": "Understandability and Coherence", "context": "Does the piece hold together? In other words, does the beginning lead through the middle to the end in a way that feels deliberate and intentional? This is the difference between several pages of writing and a PIECE of writing. Great writing errs on the side of unity over disorder.", "question": "Do the different elements of the story work together to form a unified, engaging, and satisfying whole?"},
    {"aspect_id": 6, "category": "flexibility", "aspect_name": "Perspective and Voice Flexibility", "context": "A good writer can convincingly and accurately depict a wide range of character viewpoints, including those of characters who may be morally ambiguous, difficult, or otherwise unappealing.", "question": "Does the story inhabit various perspectives, even unlikable ones?"},
    {"aspect_id": 7, "category": "flexibility", "aspect_name": "Emotional Flexibility", "context": "'Emotional flexibility' is asking whether the piece of writing effectively balances action and introspection, and if it portrays a broad and realistic spectrum of emotions.", "question": "Is there a balance between exteriority and interiority? Emotional flexibility."},
    {"aspect_id": 8, "category": "flexibility", "aspect_name": "Structural Flexibility", "context": "A writer must be able to make turns that are both surprising and appropriate to strike a balance between unexpectedness and coherence, keeping the reader on their toes while maintaining a believable, satisfying narrative.", "question": "Does the story contain turns that are both surprising and appropriate?"},
    {"aspect_id": 9, "category": "originality", "aspect_name": "Originality in Theme and Content", "context": "If a story is good, the reader gains new insights, perspectives, or knowledge from it . This doesn't necessarily mean factual information, but could relate to a deeper understanding of human nature, cultural insights, unique viewpoints, or even the exploration of new ideas and themes. Essentially, it's about what the reader takes away from the story beyond just the plot", "question": "Do we learn something new from the story a.k.a is there a  purpose of putting this story into the world?"},
    {"aspect_id": 10, "category": "originality", "aspect_name": "Originality in Thought", "context": "A cliche is an idea, expression, character, or plot that has been overused to the point of losing its original meaning or impact. They often become predictable and uninteresting for the reader. Originality suggests that the piece isn't cliche.", "question": "Is the story an original piece of writing without any cliches?"},
    {"aspect_id": 11, "category": "originality", "aspect_name": "Originality in Form and Structure", "context": "Innovative use of form/structure refers to when writers choose to shape and organize the story in an unusual, original, or inventive way, involving elements such as unconventional timelines, multiple perspectives, or unique narrative voices.", "question": "Does the story show originality in its form and/or structure?"},
    {"aspect_id": 12, "category": "elaboration", "aspect_name": "Character Development", "context": "In good stories authors take a character who initially appears to be one-dimensional or stereotypical (flat), and add depth to them. This could be done by revealing more about their backstory, introducing unexpected traits or motivations, or having them grow and change in response to the events of the story.", "question": "Does the piece make a flat character complex?"},
    {"aspect_id": 13, "category": "elaboration", "aspect_name": "Rhetorical Complexity", "context": "Effective fiction can operate at both a surface and subtext level. The surface text keeps the reader engaged with the plot and characters, while the subtext provides depth, complexity, and additional layers of interpretation, contributing to a richer and more rewarding reading experience.", "question": "Does the story operate at multiple 'levels' of meaning (surface and subtext)?"},
    {"aspect_id": 14, "category": "elaboration", "aspect_name": "World Building and Setting", "context": "Sensory details pertain to the five senses - sight, sound, touch, taste, and smell. An effective writer can use these elements to paint a detailed picture of the story's environment, making it feel tangible and real to the reader.", "question": "Does the writer make the fictional world believable at the sensory level?"},
]

ttcw_instruction = "Write a New Yorker style fiction given the plot below. Make sure it is atleast 1500 words. Directly start with the story, do not say things like `Here's the story [...]:`\n\nPlot:"

story_folder = "data/art_or_artifice/teststories/"
final_dataset = []
for folder in os.listdir(story_folder):
    with open(f"{story_folder}/{folder}/plot.txt", "r") as f:
        plot = f.read()

    full_instruction = ttcw_instruction + "\n" + plot
    final_dataset.append({"instruction": full_instruction, "reference_aspects": ttcw_aspects})

with open("data/art_or_artifice/art_or_artifice_reference_aspects.json", "w") as f:
    json.dump(final_dataset, f, indent=4)
